# Single LST-Stack Baseline Scaffolded and Feathered 1D DPSS (Re-)Inpainting

**by Josh Dillon**, last updated May 13, 2026

For one LST-stack single-baseline cross file, re-inpaints pixels that are either `nsamples == 0`
or flagged by the round-6 RFI step, inside the `[BAND_STR, LST_MIN, LST_MAX]` analysis window.
The DPSS fit uses inverse-variance weights from the smoothed auto-correlation (reused via a
shared `.npz` cache with the SNR step), with `nsamples` taken as band-constant (separately above
and below FM) for the fit weights only. Re-inpainted pixels are written with `nsamples = 0`.
Optionally multiplies the cross visibilities by an SSM gleam/orig abscal ratio before writing.
The sibling auto file on disk is read but never modified — auto re-inpainting and any auto
abscal correction are handled by a separate job.

Based on the per-night [`single_baseline_scaffolded_and_feathered_inpainter.ipynb`](https://github.com/HERA-Team/hera_notebook_templates/blob/master/notebooks/single_baseline_scaffolded_and_feathered_inpainter.ipynb), restructured for one-baseline LST-stack use.

In [ ]:
import time
tstart = time.time()
!hostname
!date

In [ ]:
import os
os.environ['HDF5_USE_FILE_LOCKING'] = 'FALSE'
from pathlib import Path
import re
import copy
import hashlib
import tempfile
import toml
import h5py
import hdf5plugin
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from scipy import interpolate
from pyuvdata import UVFlag
from hera_cal import io, utils, flag_utils
from hera_cal.frf import get_FR_buffer_from_spectra
from hera_filters.dspec import dpss_operator, fourier_filter, sparse_linear_fit_2D
from hera_notebook_templates.data import DATA_PATH as HNBT_DATA
from IPython.display import display
%matplotlib inline
_ = np.seterr(all='ignore')
%config InlineBackend.figure_format = 'retina'

In [ ]:
# Path-style settings come in as env vars (set by the bash wrapper).
TOML_FILE: str = os.environ.get('TOML_FILE',
    '/lustre/aoc/projects/hera/h6c-analysis/IDR3/src/hera_pipelines/pipelines/h6c/idr3/v1/post-lstbin/h6c_pspec_11band.toml')
SINGLE_BL_FILE: str = os.environ.get('SINGLE_BL_FILE',
    '/lustre/aoc/projects/hera/h6c-analysis/IDR3/pspec-outputs-131-nights/zen.LST.baseline.0_4.sum.uvh5')
OUT_FILE: str = os.environ.get('OUT_FILE',
    str(Path(SINGLE_BL_FILE).with_suffix('.reinpainted.uvh5')))
# Scaffold defaults to the input itself (1D DPSS fit uses input data values where flagged).
SCAFFOLD_FILE: str = os.environ.get('SCAFFOLD_FILE', SINGLE_BL_FILE)

for setting in ['TOML_FILE', 'SINGLE_BL_FILE', 'OUT_FILE', 'SCAFFOLD_FILE']:
    print(f'{setting} = "{eval(setting)}"')

# Load all relevant knobs from the TOML and inject into globals.
toml_options = toml.load(TOML_FILE)
for toml_section in ['GLOBAL_OPTS', 'AUTO_SMOOTH_OPTS', 'SINGLE_LSTSTACK_BASELINE_REINPAINT_OPTS']:
    print(f'\nLoading config from [{toml_section}] in {TOML_FILE}:')
    for key, val in toml_options[toml_section].items():
        globals()[key.upper()] = val
        print(f'  {key.upper()} = {val!r}')

SINGLE_BL_FILE = Path(SINGLE_BL_FILE)
OUT_FILE = Path(OUT_FILE)
SCAFFOLD_FILE = Path(SCAFFOLD_FILE)
AUTO_FR_SPECTRUM_FILE = Path(AUTO_FR_SPECTRUM_FILE)

CG_ITER_LIM = int(CG_ITER_LIM)
print(f'\nResolved: APPLY_ROUND_6_FLAGS={APPLY_ROUND_6_FLAGS}, APPLY_ABSCAL_CORRECTION={APPLY_ABSCAL_CORRECTION}')

In [ ]:
add_to_history = (
    'Produced by single_lststack_baseline_scaffolded_and_feathered_inpainter notebook '
    f'with TOML_FILE={TOML_FILE} and the following environment:\n'
    + '=' * 65 + '\n' + os.popen('mamba env export').read() + '=' * 65
)

## Load cross + auto data; restrict to (BAND_STR, LST_MIN/MAX)

In [ ]:
# figure out ANTPAIR and the sibling auto file (i==j) via parent-glob
ANTPAIR = tuple(int(a) for a in re.search(r'(\d+)_(\d+)', SINGLE_BL_FILE.name).group().split('_'))
glob_pattern = SINGLE_BL_FILE.parent / SINGLE_BL_FILE.name.replace(f'{ANTPAIR[0]}_{ANTPAIR[1]}', '*')
auto_candidates = sorted(SINGLE_BL_FILE.parent.glob(glob_pattern.name))
AUTO_BL_FILE = next(f for f in auto_candidates
                    if (m := re.search(r'(\d+)_(\d+)', f.name))
                    and m.group(1) == m.group(2))
print(f'Loading cross from {SINGLE_BL_FILE.name}')
print(f'Loading auto from  {AUTO_BL_FILE.name}')

# Load both (reading times only from the cross, applied to the auto too)
single_bl_times = np.array(io.HERAData(str(SINGLE_BL_FILE)).times)
hd = io.HERAData([str(SINGLE_BL_FILE), str(AUTO_BL_FILE)])
data, flags, nsamples = hd.read(times=single_bl_times)

# Split into cross + auto views
cross_bls = [ANTPAIR + (pol,) for pol in data.pols()]
auto_antpair = sorted(set([k[:2] for k in data.bls() if k[0] == k[1]]))[0]
auto_bls = [auto_antpair + (pol,) for pol in data.pols()]

dt = np.median(np.diff(data.times)) * 24 * 3600  # seconds
df = np.median(np.diff(data.freqs))               # Hz
print(f'\nantpair = {ANTPAIR}, auto antpair = {auto_antpair}')
print(f'dt = {dt:.3f} s, df = {df / 1e6:.4f} MHz, shape = ({len(data.times)}, {len(data.freqs)})')

# Build the (band, LST) window mask (used both as a flag stamp and to gate where round-6 applies).
bands_MHz = [tuple(float(e) for e in b.split('~')) for b in BAND_STR.strip().split(',')]
freqs_MHz = data.freqs / 1e6
in_any_band = np.zeros(len(freqs_MHz), dtype=bool)
for lo, hi in bands_MHz:
    in_any_band |= (freqs_MHz >= lo) & (freqs_MHz <= hi)
lsts_h = np.where(data.lsts > data.lsts[-1], data.lsts - 2 * np.pi, data.lsts) * 12 / np.pi
in_lst_range = (lsts_h >= LST_MIN) & (lsts_h <= LST_MAX)
in_window = in_lst_range[:, None] & in_any_band[None, :]
print(f'\nin_window: {np.mean(in_window):.3%} of waterfall '
      f'({np.sum(in_lst_range)}/{len(in_lst_range)} integrations, '
      f'{np.sum(in_any_band)}/{len(in_any_band)} channels)')

# Completely flag everything outside the (BAND_STR, LST_MIN/MAX) window of interest.
for bl in data:
    flags[bl] |= ~in_window

# tslices/bands: split at FM midpoint, restricted to in-window pixels.
tslices_bl, bands_bl = flag_utils.get_minimal_slices(flags[ANTPAIR + ('ee',)] | flags[ANTPAIR + ('nn',)], 
                                                     freqs=data.freqs, freq_cuts=[(FM_LOW_FREQ + FM_HIGH_FREQ) * .5e6])
print(f'tslices = {tslices_bl}')
print(f'bands = {bands_bl}')

In [ ]:
# Default: data is its own scaffold (which makes the np.where(flag, scaffold, data) merge a no-op).
# If a separate SCAFFOLD_FILE is provided, load it and validate it covers all unflagged data.
if str(SCAFFOLD_FILE) == str(SINGLE_BL_FILE):
    print('Using input data as its own scaffold.')
    scaffold = data
    scaffold_flags = flags
else:
    print(f'Loading scaffold from {SCAFFOLD_FILE}')
    hd_scaffold = io.HERAData(str(SCAFFOLD_FILE))
    scaffold, scaffold_flags, _ = hd_scaffold.read(times=single_bl_times)
    for bl in cross_bls:
        if np.any(~flags[bl] & scaffold_flags[bl]):
            raise ValueError(f'Scaffold is flagged at {np.sum(~flags[bl] & scaffold_flags[bl])} locations ' 
                             f'where data is unflagged for {bl}. The scaffold must cover all unflagged data.')

## Round-6 flags and the to-reinpaint mask

In [ ]:
# Glob *.flag_waterfall_round_6.h5 next to the cross file and OR them all together.
round6_flag_files = sorted(SINGLE_BL_FILE.parent.glob('*.flag_waterfall_round_6.h5'))
print(f'Found {len(round6_flag_files)} round-6 flag file(s):')
for f in round6_flag_files:
    print(f'  {f.name}')

round6_flags = np.zeros((len(data.times), len(data.freqs)), dtype=bool)
if APPLY_ROUND_6_FLAGS and round6_flag_files:
    for f in round6_flag_files:
        uvf = UVFlag(str(f))
        # UVFlag waterfall mode: flag_array has shape (Ntimes, Nfreqs[, Npols])
        arr = uvf.flag_array
        if arr.ndim == 3:
            arr = np.any(arr, axis=-1)
        round6_flags |= arr.astype(bool)
    print(f'OR-combined round-6 flags: {np.mean(round6_flags):.3%} of waterfall flagged.')
elif not APPLY_ROUND_6_FLAGS:
    print('APPLY_ROUND_6_FLAGS is False; only re-inpainting nsamples == 0 pixels.')

In [ ]:
# to_reinpaint[bl]: pixels we will inpaint over.
#   - nsamples == 0 anywhere
#   - round-6 flagged AND inside the (BAND_STR, LST) window
# We additionally OR the to_reinpaint mask into the data flags so the DPSS fit weights them at 0.
to_reinpaint = {}
for bl in cross_bls:
    to_reinpaint[bl] = (nsamples[bl] == 0) | (round6_flags & in_window)
    flags[bl] |= to_reinpaint[bl]
    n_re = int(np.sum(to_reinpaint[bl]))
    n_re_in = int(np.sum(to_reinpaint[bl] & in_window))
    n_in = int(np.sum(in_window))
    print(f'  {bl}: {n_re} pixels marked for reinpainting '
          f'({n_re_in}/{n_in} = {n_re_in / max(n_in,1):.2%} inside the window).')

## Smooth autocorrelations (reuse SNR cache if present)

2-stage DPSS smoothing identical to the SNR step. Cache key = MD5(toml_dump + auto filename); both
notebooks read the same `[AUTO_SMOOTH_OPTS]` section, so an SNR-cached `.npz` is reused verbatim
here. If absent, compute and atomically write so parallel jobs converge.

In [ ]:
def smooth_one_auto(auto_wf, nsamp_wf, dc, tslices, bands,
                    inpaint_delay_ns=AUTO_INPAINT_DELAY,
                    blacklist_rel_err_thresh=AUTO_BLACKLIST_REL_ERR_THRESH,
                    blacklist_smoother_factor=AUTO_BLACKLIST_SMOOTHER_FACTOR,
                    blacklist_downweight=AUTO_BLACKLIST_DOWNWEIGHT,
                    eigenval_cutoff=EIGENVAL_CUTOFF,
                    cg_tol=AUTO_CG_TOL,
                    cg_iter_lim=AUTO_CG_ITER_LIM):
    '''Two-stage DPSS smoothing of one auto polarization. See single_lststack_baseline_pI_FRF_SNR
    notebook (smooth-autos cell) for the algorithm details. Returns (smooth, blacklist).'''
    smooth = np.full_like(auto_wf, np.nan, dtype=complex)
    blacklist = np.zeros(auto_wf.shape, dtype=bool)
    times_s = (dc.times - dc.times[0]) * 86400.0
    freqs_hz = dc.freqs

    for tslice, band in zip(tslices, bands):
        if (tslice is None) or (band is None):
            continue
        fr_buffer = get_FR_buffer_from_spectra(
            str(AUTO_FR_SPECTRUM_FILE), dc.times[tslice], freqs_hz[band],
            gauss_fit_buffer_cut=AUTO_GAUSS_FIT_BUFFER_CUT,
        )
        fr_hw_hz = np.max(fr_buffer) / 1e3

        with np.errstate(invalid='ignore', divide='ignore'):
            weights = np.where(nsamp_wf[tslice, band] > 0,
                               nsamp_wf[tslice, band] / auto_wf[tslice, band]**2,
                               0).astype(float)
        if not np.any(weights > 0):
            continue

        coarse_freq_basis, _ = dpss_operator(
            freqs_hz[band], [0.0], [inpaint_delay_ns * blacklist_smoother_factor / 1e9],
            eigenval_cutoff=[eigenval_cutoff],
        )
        coarse_time_basis, _ = dpss_operator(
            times_s[tslice], [0.0], [fr_hw_hz * blacklist_smoother_factor],
            eigenval_cutoff=[eigenval_cutoff],
        )
        coarse_coeffs_v0, _ = sparse_linear_fit_2D(
            data=auto_wf[tslice, band], weights=weights,
            axis_1_basis=coarse_time_basis, axis_2_basis=coarse_freq_basis,
            precondition_solver=True, atol=cg_tol, btol=cg_tol, iter_lim=cg_iter_lim,
        )
        coarse_v0 = np.abs(coarse_time_basis @ coarse_coeffs_v0 @ coarse_freq_basis.T)

        with np.errstate(invalid='ignore', divide='ignore'):
            rel_err = np.abs(auto_wf[tslice, band] - coarse_v0) / np.abs(coarse_v0)
        bl_local = (np.isfinite(rel_err) & (rel_err > blacklist_rel_err_thresh)
                    & (nsamp_wf[tslice, band] > 0))
        blacklist[tslice, band] = bl_local

        weights_coarse_v1 = weights * np.where(bl_local, blacklist_downweight, 1.0)
        if not np.any(weights_coarse_v1 > 0):
            smooth[tslice, band] = coarse_v0
            continue
        coarse_coeffs_v1, _ = sparse_linear_fit_2D(
            data=auto_wf[tslice, band], weights=weights_coarse_v1,
            axis_1_basis=coarse_time_basis, axis_2_basis=coarse_freq_basis,
            precondition_solver=True, atol=cg_tol, btol=cg_tol, iter_lim=cg_iter_lim,
        )
        coarse_v1 = np.abs(coarse_time_basis @ coarse_coeffs_v1 @ coarse_freq_basis.T)

        data_final = np.where(bl_local, coarse_v1, auto_wf[tslice, band])
        weights_final = weights * np.where(bl_local, blacklist_downweight, 1.0)
        freq_basis, _ = dpss_operator(freqs_hz[band], [0.0], [inpaint_delay_ns / 1e9],
                                       eigenval_cutoff=[eigenval_cutoff])
        time_basis, _ = dpss_operator(times_s[tslice], [0.0], [fr_hw_hz],
                                       eigenval_cutoff=[eigenval_cutoff])
        final_coeffs, _ = sparse_linear_fit_2D(
            data=data_final, weights=weights_final,
            axis_1_basis=time_basis, axis_2_basis=freq_basis,
            precondition_solver=True, atol=cg_tol, btol=cg_tol, iter_lim=cg_iter_lim,
        )
        smooth[tslice, band] = np.abs(time_basis @ final_coeffs @ freq_basis.T)

    return smooth, blacklist

# Cache key: TOML + auto file basename. Shared with single_lststack_baseline_pI_FRF_SNR.
cache_key = hashlib.md5((toml.dumps(toml_options) + AUTO_BL_FILE.name).encode()).hexdigest()
auto_cache_file = OUT_FILE.parent / f'smooth_autos_cache_{cache_key}.npz'

auto_smooth = {}
auto_blacklist = {}
if auto_cache_file.exists():
    print(f'Loading cached smoothed autos from {auto_cache_file}')
    with np.load(auto_cache_file) as cache:
        assert cache['toml_hash'].item() == cache_key, 'Cache hash mismatch'
        for pol in ('ee', 'nn'):
            auto_smooth[pol] = cache[f'auto_smooth_{pol}']
            auto_blacklist[pol] = cache[f'auto_blacklist_{pol}']
else:
    print(f'No cache found at {auto_cache_file}, smoothing autos...')
    for pol in ('ee', 'nn'):
        bl = auto_antpair + (pol,)
        print(f'Smoothing auto {bl}...')
        smooth, blk = smooth_one_auto(np.abs(data[bl]), nsamples[bl], data, tslices_bl, bands_bl)
        auto_smooth[pol] = smooth
        auto_blacklist[pol] = blk

    # Atomic write to handle parallel race conditions (parallel baselines share the auto cache).
    tmp_fd, tmp_path = tempfile.mkstemp(dir=str(OUT_FILE.parent), suffix='.npz')
    os.close(tmp_fd)
    np.savez(tmp_path,
            toml_hash=cache_key,
            **{f'auto_smooth_{pol}': auto_smooth[pol] for pol in ('ee', 'nn')},
            **{f'auto_blacklist_{pol}': auto_blacklist[pol] for pol in ('ee', 'nn')})
    try:
        os.rename(tmp_path, auto_cache_file)
        print(f'Saved smoothed autos cache to {auto_cache_file}')
    except OSError:
        os.remove(tmp_path)
        print('Cache already written by another process.')

In [ ]:
def plot_smoothed_auto(pol):
    bl = auto_antpair + (pol,)
    raw = np.abs(data[bl])
    smooth = np.abs(auto_smooth[pol])
    blk = auto_blacklist[pol]
    with np.errstate(invalid='ignore', divide='ignore'):
        rel_err = np.abs(raw - smooth) / smooth
    extent = [data.freqs[0] / 1e6, data.freqs[-1] / 1e6,
              data.times[-1] - int(data.times[0]), data.times[0] - int(data.times[0])]
    fig, axes = plt.subplots(1, 4, figsize=(20, 5), dpi=300, sharex=True, sharey=True)
    im0 = axes[0].imshow(raw, aspect='auto', cmap='viridis', extent=extent, interpolation='none',
                         norm=matplotlib.colors.LogNorm())
    axes[0].set_title(f'{pol} raw |auto|')
    plt.colorbar(im0, ax=axes[0])
    im1 = axes[1].imshow(smooth, aspect='auto', cmap='viridis', extent=extent, interpolation='none',
                         norm=matplotlib.colors.LogNorm(vmin=im0.get_clim()[0], vmax=im0.get_clim()[1]))
    axes[1].set_title(f'{pol} smooth |auto|')
    plt.colorbar(im1, ax=axes[1])
    im2 = axes[2].imshow(rel_err, aspect='auto', cmap='inferno', extent=extent, interpolation='none',
                         vmin=0, vmax=2 * AUTO_BLACKLIST_REL_ERR_THRESH)
    axes[2].set_title(f'{pol} |raw - smooth| / smooth')
    plt.colorbar(im2, ax=axes[2], extend='max')
    im3 = axes[3].imshow(blk.astype(int), aspect='auto', cmap='gray_r', extent=extent, interpolation='none',
                         vmin=0, vmax=1)
    axes[3].set_title(f'{pol} blacklist')
    plt.colorbar(im3, ax=axes[3])
    for ax in axes:
        ax.set_xlabel('Frequency (MHz)')
    axes[0].set_ylabel(f'JD - {int(data.times[0])}')
    plt.tight_layout()
    plt.show()


## *Figure 1: Raw vs smoothed autocorrelations*

In [ ]:
for pol in ('ee', 'nn'):
    plot_smoothed_auto(pol)

## Band-constant nsamples for fit weights

`n_eff[pol]` is the per-integration median of nsamples across each (above-FM / below-FM) band,
broadcast back across that band. Mirrors `USE_BAND_AVG_NSAMPLES` in the postproc pspec step and
the `n_*_eff` pattern in pI_FRF_SNR. Used **only** for the DPSS-fit weights; the saved nsamples
keep their real per-pixel values (set to 0 on re-inpainted pixels).

In [ ]:
n_eff = {}
for pol in data.pols():
    bl = ANTPAIR + (pol,)
    n_pol = np.zeros_like(nsamples[bl], dtype=float)
    for tslice, band in zip(tslices_bl, bands_bl):
        if (tslice is None) or (band is None):
            continue
        n_slice = nsamples[bl][tslice, band]
        with np.errstate(invalid='ignore'):
            n_med = np.nanmedian(np.where(n_slice > 0, n_slice, np.nan), axis=1, keepdims=True)
        n_med = np.where(np.isfinite(n_med), n_med, 0.0)
        n_pol[tslice, band] = np.broadcast_to(n_med, n_slice.shape)
    n_eff[pol] = n_pol
    print(f'  n_eff[{pol}]: nonzero in {np.mean(n_pol > 0):.3%} of waterfall.')

## Scaffolded feathered 1D DPSS inpainting

For each cross polarization, for each (tslice, band) covering the in-window pixels:
- compute per-integration distance to the nearest unflagged channel and convert to a
  sigmoid feathering weight that tapers flagged pixels at the flag boundary,
- build inverse-variance weights from the smoothed autos (paired up via `AUTO_POL_PAIR`)
  and the band-constant `n_eff`,
- multiply by the feathering weight where the pixel is flagged (`flags[bl]` carries both
  the input flags and the to-reinpaint mask, plus `~in_window`),
- zero the weight on integrations whose full band is flagged,
- run a 1D DPSS solve via `fourier_filter(..., mode='dpss_solve')` along frequency,
- write the DPSS model into the to-reinpaint pixels; keep the scaffold (= input data
  by default) at other flagged pixels; keep the input data at unflagged pixels.

In [ ]:
# Map each cross pol to the (auto_pol_1, auto_pol_2) whose product gives the noise model.
AUTO_POL_PAIR = {'ee': ('ee', 'ee'), 'nn': ('nn', 'nn'), 'en': ('ee', 'nn'), 'ne': ('nn', 'ee')}

ip_data = copy.deepcopy(data)
ip_flags = {bl: np.ones_like(flags[bl], dtype=bool) for bl in cross_bls}

print(f'Performing scaffolded feathered 1D DPSS inpainting out to {INPAINT_DELAY} ns.')
for bl in cross_bls:
    pol = bl[2]
    auto_p1, auto_p2 = AUTO_POL_PAIR[pol]
    a_smooth = np.sqrt(np.abs(auto_smooth[auto_p1] * auto_smooth[auto_p2]))

    # Inverse-variance weights from smoothed autos and band-constant n_eff.
    with np.errstate(invalid='ignore', divide='ignore'):
        noise_var = a_smooth**2 / (n_eff[pol] * dt * df)
        inv_var = np.where((n_eff[pol] > 0) & np.isfinite(noise_var) & (noise_var > 0),
                            1.0 / noise_var, 0.0)

    # Feathering: sigmoid weight as a function of distance to the nearest unflagged channel.
    distances = np.array([flag_utils.distance_to_nearest_nonzero(~flags[bl][t, :])
                          for t in range(flags[bl].shape[0])])
    width = (1e-9 * INPAINT_DELAY)**-1 / df * INPAINT_WIDTH_FACTOR
    rel_weights = (1 + np.exp(-np.log(INPAINT_ZERO_DIST_WEIGHT**-1 - 1) / width
                              * (distances - width)))**-1

    d_mdl = np.full_like(data[bl], np.nan)
    for tslice, band in zip(tslices_bl, bands_bl):
        if (tslice is None) or (band is None):
            continue
        # Per-pixel weight: inv_var at unflagged, inv_var * rel_weights at flagged.
        wgts = np.where(flags[bl][:, band],
                        inv_var[:, band] * rel_weights[:, band],
                        inv_var[:, band])
        # Zero out integrations that have no unflagged data in this band.
        int_fully_flagged = np.all(flags[bl][:, band], axis=1, keepdims=True)
        wgts = np.where(int_fully_flagged, 0, wgts)
        if np.any(wgts > 0):
            wgts /= np.mean(wgts[wgts > 0])

        # 1D DPSS solve along frequency; scaffold provides the model values where flagged.
        d_mdl[:, band], _, _ = fourier_filter(
            data.freqs[band],
            np.where(flags[bl], scaffold[bl], data[bl])[:, band],
            wgts=wgts,
            filter_centers=[0],
            filter_half_widths=[INPAINT_DELAY * 1e-9],
            mode='dpss_solve',
            eigenval_cutoff=[EIGENVAL_CUTOFF],
            suppression_factors=[EIGENVAL_CUTOFF],
            max_contiguous_edge_flags=len(data.freqs),
            filter_dims=1,
        )

        # Output flags: integration is flagged in the band only if it was fully flagged.
        ip_flags[bl][:, band] = np.where(int_fully_flagged, True, False)

    # Use DPSS model where we reinpainted; scaffold where flagged but not reinpainted; data otherwise.
    ip_data[bl] = np.where(to_reinpaint[bl], d_mdl,
                           np.where(flags[bl], scaffold[bl], data[bl]))

    n_re_done = int(np.sum(to_reinpaint[bl] & ~ip_flags[bl]))
    n_re_skipped = int(np.sum(to_reinpaint[bl] & ip_flags[bl]))
    print(f'  {bl}: reinpainted {n_re_done} pixels; left flagged (no fit possible) {n_re_skipped}.')

In [ ]:
# Mark re-inpainted pixels with nsamples = 0 so downstream pspec ignores them.
for bl in cross_bls:
    nsamples[bl] = np.where(to_reinpaint[bl], 0, nsamples[bl])

## Optional abscal correction

When `apply_abscal_correction = true`, compute the per-polarization ratio of the gleam-flux-scale
SSM autocorrelations to the original SSM autocorrelations, upsample to `(data.lsts, data.freqs)`
via a periodic CubicSpline over LSTs (wrap at 2π) and a CubicSpline over frequencies, then 2D-DPSS
smooth at `ABSCAL_SMOOTH_DELAY` (ns) and `ABSCAL_SMOOTH_FRINGE_RATE` (Hz) so no structure finer
than those scales is injected into the cross visibilities. Cross-pol ratios use the geometric
mean of the two single-pol ratios.

In [ ]:
if APPLY_ABSCAL_CORRECTION:
    SSM_ORIG_FILE = os.path.join(HNBT_DATA, 'SSM_autocorrelations_downsampled_sum_pol_convention.uvh5')
    SSM_GLEAM_FILE = os.path.join(HNBT_DATA, 'SSM_autocorrelations_downsampled_sum_pol_convention_gleam_flux_scale.uvh5')

    hd_ssm_orig = io.HERAData(SSM_ORIG_FILE)
    orig_model, _, _ = hd_ssm_orig.read()
    hd_ssm_gleam = io.HERAData(SSM_GLEAM_FILE)
    gleam_model, _, _ = hd_ssm_gleam.read()

    # Per-pol downsampled ratio (pair up baselines by polarization).
    ratio_per_pol_ds = {}
    for bl in orig_model:
        pol = bl[2]
        if pol in ratio_per_pol_ds or pol not in ('ee', 'nn'):
            continue
        gleam_bl = next((k for k in gleam_model if k[2] == pol), None)
        if gleam_bl is None:
            raise KeyError(f'No {pol} polarization in {SSM_GLEAM_FILE}')
        ratio_per_pol_ds[pol] = np.abs(gleam_model[gleam_bl] / orig_model[bl])

    # Upsample to (data.lsts, data.freqs) via periodic CubicSpline (LSTs) then CubicSpline (freqs),
    # then 2D DPSS-smooth at (ABSCAL_SMOOTH_FRINGE_RATE, ABSCAL_SMOOTH_DELAY) so neither the upsampling
    # nor any residual structure in the SSM ratio can inject features into the cross visibilities
    # finer than these scales.
    sorted_lsts, lst_indices = np.unique(orig_model.lsts, return_index=True)
    periodic_lsts = np.append(sorted_lsts, sorted_lsts[0] + 2 * np.pi)
    times_s = (data.times - data.times[0]) * 86400.0
    time_basis, _ = dpss_operator(times_s, [0.0], [ABSCAL_SMOOTH_FRINGE_RATE],
                                   eigenval_cutoff=[EIGENVAL_CUTOFF])
    freq_basis, _ = dpss_operator(data.freqs, [0.0], [ABSCAL_SMOOTH_DELAY / 1e9],
                                   eigenval_cutoff=[EIGENVAL_CUTOFF])
    weights_2d = np.ones((len(times_s), len(data.freqs)), dtype=float)

    auto_abscal_ratio = {}
    for pol, ratio_ds in ratio_per_pol_ds.items():
        periodic_ratio = np.vstack([ratio_ds[lst_indices, :], ratio_ds[lst_indices[0], :]])
        lst_interp = interpolate.CubicSpline(periodic_lsts, periodic_ratio, axis=0,
                                              bc_type='periodic')(data.lsts)
        freq_interp = interpolate.CubicSpline(orig_model.freqs, lst_interp, axis=1)(data.freqs)
        coeffs, _ = sparse_linear_fit_2D(
            data=freq_interp,
            weights=weights_2d,
            axis_1_basis=time_basis,
            axis_2_basis=freq_basis,
            precondition_solver=True,
            atol=AUTO_CG_TOL, btol=AUTO_CG_TOL, iter_lim=AUTO_CG_ITER_LIM,
        )
        auto_abscal_ratio[pol] = time_basis @ coeffs @ freq_basis.T

    # Cross-pol ratio = geometric mean of the two single-pol ratios (per-antenna gain factorization).
    abscal_ratio = {
        'ee': auto_abscal_ratio['ee'],
        'nn': auto_abscal_ratio['nn'],
        'en': np.sqrt(auto_abscal_ratio['ee'] * auto_abscal_ratio['nn']),
        'ne': np.sqrt(auto_abscal_ratio['ee'] * auto_abscal_ratio['nn']),
    }
    for bl in cross_bls:
        ip_data[bl] = ip_data[bl] * abscal_ratio[bl[2]]
    print(f'Applied upsampled + 2D-DPSS-smoothed SSM gleam/orig abscal ratio '
          f'(delay={ABSCAL_SMOOTH_DELAY} ns, fringe rate={ABSCAL_SMOOTH_FRINGE_RATE} Hz) '
          f'to cross visibilities.')

## Write the re-inpainted cross to disk

In [ ]:
# Restrict hd to the cross baselines for writing.
hd_out = io.HERAData(str(SINGLE_BL_FILE))
data_out, flags_out, nsamples_out = hd_out.read()
for bl in cross_bls:
    data_out[bl] = ip_data[bl]
    flags_out[bl] = ip_flags[bl]
    nsamples_out[bl] = nsamples[bl]
hd_out.update(data=data_out, flags=flags_out, nsamples=nsamples_out)
hd_out.history += add_to_history
print(f'Writing re-inpainted cross to {OUT_FILE}')
hd_out.write_uvh5(str(OUT_FILE), clobber=True)

In [ ]:
def four_pol_inpainting_figure():
    '''Mirror of the per-night reinpainter's four_pol_inpainting_figure: one row per cross pol,
    by 4 columns (phase before/after, |V| before/after on a shared LogNorm). Flagged input pixels
    are rendered as NaN in the "before" columns, ip_flags as NaN in the "after" columns.'''
    npols = len(cross_bls)
    fig, axes = plt.subplots(npols, 4, figsize=(16, 7.5 * npols), sharex=True, sharey=True, dpi=200,
                             gridspec_kw={'wspace': 0.02, 'hspace': 0.01}, squeeze=False)

    vmin = np.nanmin([np.where(~flags[bl] & (np.abs(ip_data[bl]) > 0), np.abs(ip_data[bl]), np.nan)
                      for bl in cross_bls])
    vmax = np.nanmax([np.where(~flags[bl] & np.isfinite(ip_data[bl]), np.abs(ip_data[bl]), np.nan)
                      for bl in cross_bls])
    lst_grid = data.lsts * 12 / np.pi
    lst_grid[lst_grid > lst_grid[-1]] -= 24
    frac_jds = data.times - int(data.times[0])
    extent = [data.freqs[0] / 1e6, data.freqs[-1] / 1e6, frac_jds[-1], frac_jds[0]]

    for row, bl in zip(axes, cross_bls):
        row[0].imshow(np.where(flags[bl], np.nan, np.angle(ip_data[bl])),
                       aspect='auto', interpolation='none', cmap='twilight', extent=extent)
        row[1].imshow(np.where(ip_flags[bl], np.nan, np.angle(ip_data[bl])),
                       aspect='auto', interpolation='none', cmap='twilight', extent=extent)
        row[2].imshow(np.where(flags[bl], np.nan, np.abs(ip_data[bl])),
                       aspect='auto', interpolation='none',
                       norm=matplotlib.colors.LogNorm(vmin=vmin, vmax=vmax), extent=extent)
        im = row[3].imshow(np.where(ip_flags[bl], np.nan, np.abs(ip_data[bl])),
                           aspect='auto', interpolation='none',
                           norm=matplotlib.colors.LogNorm(vmin=vmin, vmax=vmax), extent=extent)

        row[0].set_ylabel(f'JD - {int(data.times[0])}')
        for ax in row:
            ax.tick_params(axis='x', direction='in')

        row[0].text(0.02, 0.99, bl, transform=row[0].transAxes, ha='left', va='top',
                    fontsize=12, color='white',
                    bbox=dict(facecolor='black', alpha=0.5, pad=2))

    # LST right axis on the last row's rightmost panel
    ax2 = axes[-1][-1].twinx()
    ax2.set_ylim(lst_grid[-1], lst_grid[0])
    mod24 = lambda x, _: f"{x % 24:.1f}"
    ax2.yaxis.set_major_formatter(matplotlib.ticker.FuncFormatter(mod24))
    ax2.set_ylabel('LST (hours)')

    for ax in axes[-1]:
        ax.set_xlabel('Frequency (MHz)')

    plt.tight_layout()
    plt.colorbar(im, ax=axes[0], location='top', label='|V| (Jy)', aspect=50)
    plt.show()

# *Figure 2: 4-Pol Phase and Amplitude Waterfalls Before and After Inpainting*

In [ ]:
four_pol_inpainting_figure()

## Metadata

In [ ]:
for repo in ['hera_cal', 'hera_qm', 'hera_filters', 'hera_notebook_templates', 'pyuvdata', 'numpy']:
    exec(f'from {repo} import __version__')
    print(f'{repo}: {__version__}')

In [ ]:
print(f'Finished execution in {(time.time() - tstart) / 60:.2f} minutes.')